In [1]:

import torch
import yaml
from CHILI_centralAtoms import CHILI
from benchmark import validate_dataset_atom_count
from torch_geometric.loader import DataLoader
from test_vectordiff_overfitting import create_small_dataset
from mnist_ddpm_cond import train_vector_conditioned_ddpm

In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
with open('configs/diffusion_xpdf_abs_base.yaml', "r") as file:
    config = yaml.safe_load(file)
dataset_config = config['dataset']
dataset = CHILI(root=dataset_config["root"], dataset=dataset_config["name"], graph_type=dataset_config["graph_type"])

try:
    dataset.load_data_split(split_strategy = 'random')
except FileNotFoundError:
    print("No data split found, first run create_data_split")

print(dataset)


CHILI(3160)


In [4]:
# Create dataloaders
train_loader = DataLoader(
    dataset.train_set,
    batch_size=config["Train_config"]["batch_size"],
    shuffle=True,
)
val_loader = DataLoader(
    dataset.validation_set,
    batch_size=config["Train_config"]["batch_size"],
    shuffle=False,
)
test_loader = DataLoader(
    dataset.test_set,
    batch_size=config["Train_config"]["batch_size"],
    shuffle=False,
)

In [5]:
from run_benchmarks import run_experiment

model, metrics = run_experiment('configs/diffusion_xpdf_frac_base.yaml')


2025-05-13 20:08:32,239 - INFO - Starting experiment with model: Diffusion
2025-05-13 20:08:32,240 - INFO - SLURM Job ID: local, Task ID: 0
2025-05-13 20:08:32,240 - INFO - Task type: FracPositionRegressionXPDF
2025-05-13 20:08:32,241 - INFO - No specific GPU assignment, using all 1 available GPUs
2025-05-13 20:08:32,244 - INFO - GPU 0: NVIDIA GeForce RTX 3060 Laptop GPU
2025-05-13 20:08:32,245 - INFO - Using device: cuda
2025-05-13 20:08:32,248 - INFO - Using random seed: 42
2025-05-13 20:08:32,248 - INFO - Loading dataset from data/CHILI-3K
2025-05-13 20:08:32,271 - INFO - Dataset: CHILI-3K
2025-05-13 20:08:32,271 - INFO - Train samples: 2031
2025-05-13 20:08:32,272 - INFO - Validation samples: 255
2025-05-13 20:08:32,272 - INFO - Test samples: 255
2025-05-13 20:08:32,273 - INFO - Results will be saved to: diffusion_results\FracPositionRegressionXPDF_20250513_200832_joblocal_task0
2025-05-13 20:08:32,275 - INFO - Starting diffusion model training...
2025-05-13 20:08:32,276 - INFO - U

Loaded atom mapping with 54 categories for visualization
Loaded atom mapping with 54 categories
Created dataset with 2031 samples, image shape: torch.Size([2031, 3, 10, 10]), conditioning shape: torch.Size([2031, 6000])
Atom types shape: torch.Size([2031, 1, 10, 10]), with 54 categories
Loaded atom mapping with 54 categories
Created dataset with 255 samples, image shape: torch.Size([255, 3, 10, 10]), conditioning shape: torch.Size([255, 6000])
Atom types shape: torch.Size([255, 1, 10, 10]), with 54 categories
Loaded atom mapping with 54 categories
Created dataset with 255 samples, image shape: torch.Size([255, 3, 10, 10]), conditioning shape: torch.Size([255, 6000])
Atom types shape: torch.Size([255, 1, 10, 10]), with 54 categories
Using device: cuda:0


Training:   0%|          | 0/3200 [00:00<?, ?it/s]


Epoch 1/100 - Train Loss: 4.7154, Val MAE: 3.6483, Val Hausdorff: 6.0529


KeyboardInterrupt: 

In [9]:
# Train model with updated loss
model = train_vector_conditioned_ddpm(
    T=100,
    learning_rate=1e-3,
    train_data=train_loader,
    val_data=val_loader,
    test_data=test_loader,
    epochs=100,
    batch_size=64,
    ema=True,
    model_type='pos_frac',
    atom_mapping_path='atom_type_mapping.json',
    cond_type='xPDF',
    sample_dir='test_model_samples'
)

Loaded atom mapping with 54 categories for visualization
Loaded atom mapping with 54 categories
Created dataset with 2031 samples, image shape: torch.Size([2031, 3, 10, 10]), conditioning shape: torch.Size([2031, 6000])
Atom types shape: torch.Size([2031, 1, 10, 10]), with 54 categories
Loaded atom mapping with 54 categories
Created dataset with 255 samples, image shape: torch.Size([255, 3, 10, 10]), conditioning shape: torch.Size([255, 6000])
Atom types shape: torch.Size([255, 1, 10, 10]), with 54 categories
Loaded atom mapping with 54 categories
Created dataset with 255 samples, image shape: torch.Size([255, 3, 10, 10]), conditioning shape: torch.Size([255, 6000])
Atom types shape: torch.Size([255, 1, 10, 10]), with 54 categories
Using device: cuda:0


Training:   0%|          | 0/3200 [00:00<?, ?it/s]


Epoch 1/100 - Train Loss: 3.8210, Val MAE: 3.5094, Val Hausdorff: 5.7983

Epoch 2/100 - Train Loss: 3.1369, Val MAE: 3.5825, Val Hausdorff: 5.9132

Epoch 3/100 - Train Loss: 3.0941, Val MAE: 3.4648, Val Hausdorff: 5.5758

Epoch 4/100 - Train Loss: 3.0569, Val MAE: 3.5162, Val Hausdorff: 5.8022

Epoch 5/100 - Train Loss: 3.0547, Val MAE: 3.3816, Val Hausdorff: 5.4506

Epoch 6/100 - Train Loss: 2.9654, Val MAE: 2.3750, Val Hausdorff: 3.3498


KeyboardInterrupt: 

In [7]:
from run_benchmarks import run_experiment

model, metrics = run_experiment('configs/diffusion_xpdf_frac_base.yaml')

2025-05-13 21:08:00,613 - INFO - Starting experiment with model: Diffusion
2025-05-13 21:08:00,614 - INFO - SLURM Job ID: local, Task ID: 0
2025-05-13 21:08:00,614 - INFO - Task type: FracPositionRegressionXPDF
2025-05-13 21:08:00,615 - INFO - No specific GPU assignment, using all 1 available GPUs
2025-05-13 21:08:00,615 - INFO - GPU 0: NVIDIA GeForce RTX 3060 Laptop GPU
2025-05-13 21:08:00,616 - INFO - Using device: cuda
2025-05-13 21:08:00,619 - INFO - Using random seed: 42
2025-05-13 21:08:00,620 - INFO - Loading dataset from data/CHILI-3K
2025-05-13 21:08:00,643 - INFO - Dataset: CHILI-3K
2025-05-13 21:08:00,643 - INFO - Train samples: 2031
2025-05-13 21:08:00,644 - INFO - Validation samples: 255
2025-05-13 21:08:00,644 - INFO - Test samples: 255
2025-05-13 21:08:00,645 - INFO - Results will be saved to: diffusion_results\FracPositionRegressionXPDF_20250513_210800_joblocal_task0
2025-05-13 21:08:00,648 - INFO - Starting diffusion model training...
2025-05-13 21:08:00,649 - INFO - U

Loaded atom mapping with 54 categories for visualization
Loaded atom mapping with 54 categories
Created dataset with 2031 samples, image shape: torch.Size([2031, 3, 10, 10]), conditioning shape: torch.Size([2031, 6000])
Atom types shape: torch.Size([2031, 1, 10, 10]), with 54 categories
Loaded atom mapping with 54 categories
Created dataset with 255 samples, image shape: torch.Size([255, 3, 10, 10]), conditioning shape: torch.Size([255, 6000])
Atom types shape: torch.Size([255, 1, 10, 10]), with 54 categories
Loaded atom mapping with 54 categories
Created dataset with 255 samples, image shape: torch.Size([255, 3, 10, 10]), conditioning shape: torch.Size([255, 6000])
Atom types shape: torch.Size([255, 1, 10, 10]), with 54 categories
Using device: cuda:0


Training:   0%|          | 0/3200 [00:00<?, ?it/s]


Epoch 1/100 - Train Loss: 4.7154, Val MAE: 3.6483, Val Hausdorff: 6.0529

Epoch 2/100 - Train Loss: 3.5939, Val MAE: 3.5535, Val Hausdorff: 5.8388

Epoch 3/100 - Train Loss: 3.2888, Val MAE: 3.4972, Val Hausdorff: 5.6198

Epoch 4/100 - Train Loss: 3.1561, Val MAE: 3.3106, Val Hausdorff: 5.4737

Epoch 5/100 - Train Loss: 3.0525, Val MAE: 3.0267, Val Hausdorff: 4.8366

Epoch 6/100 - Train Loss: 2.9547, Val MAE: 2.6791, Val Hausdorff: 4.1742

Epoch 7/100 - Train Loss: 2.8770, Val MAE: 2.3829, Val Hausdorff: 3.6072

Epoch 8/100 - Train Loss: 2.8238, Val MAE: 2.2479, Val Hausdorff: 3.3166

Epoch 9/100 - Train Loss: 2.7771, Val MAE: 2.0763, Val Hausdorff: 2.8349

Epoch 10/100 - Train Loss: 2.7467, Val MAE: 2.0179, Val Hausdorff: 2.7729

Epoch 11/100 - Train Loss: 2.7301, Val MAE: 1.9498, Val Hausdorff: 2.6107

Epoch 12/100 - Train Loss: 2.7064, Val MAE: 1.9112, Val Hausdorff: 2.5291

Epoch 13/100 - Train Loss: 2.6940, Val MAE: 1.8415, Val Hausdorff: 2.3387

Epoch 14/100 - Train Loss: 2.6798

2025-05-13 23:12:04,630 - INFO - Training completed in 7443.98 seconds
2025-05-13 23:12:04,631 - INFO - Final validation MAE: 1.8537
2025-05-13 23:12:04,632 - INFO - Test MAE: 1.8728
2025-05-13 23:12:04,632 - INFO - Test Hausdorff: 2.2194
2025-05-13 23:12:04,633 - INFO - Experiment completed successfully!


Test MAE: 1.8728, Test Hausdorff: 2.2194
Final test MAE: 1.8728, Test Hausdorff: 2.2194
Final metrics saved to diffusion_results\FracPositionRegressionXPDF_20250513_210800_joblocal_task0\samples\training_samples\pos_frac_T100_lr0.0001_epochs100_batch64_cond64_with_atoms_20250513-210800\final_metrics.csv


In [6]:
# Train the model
model = train_vector_conditioned_ddpm(
    T=100,
    learning_rate=1e-3,
    train_data=train_loader,
    val_data=val_loader,
    test_data=test_loader,
    epochs=100,
    batch_size=64,
    ema=True,
    model_type='pos_frac',
    atom_mapping_path='atom_type_mapping.json',
    cond_type='xPDF',
    sample_dir='test_model_samples'
)

Loaded atom mapping with 54 categories for visualization
Loaded atom mapping with 54 categories
Created dataset with 2031 samples, image shape: torch.Size([2031, 3, 10, 10]), conditioning shape: torch.Size([2031, 6000])
Atom types shape: torch.Size([2031, 1, 10, 10]), with 54 categories
Loaded atom mapping with 54 categories
Created dataset with 255 samples, image shape: torch.Size([255, 3, 10, 10]), conditioning shape: torch.Size([255, 6000])
Atom types shape: torch.Size([255, 1, 10, 10]), with 54 categories
Loaded atom mapping with 54 categories
Created dataset with 255 samples, image shape: torch.Size([255, 3, 10, 10]), conditioning shape: torch.Size([255, 6000])
Atom types shape: torch.Size([255, 1, 10, 10]), with 54 categories
Using device: cuda:0


Training:   0%|          | 0/3200 [00:00<?, ?it/s]


Epoch 1/100 - Train Loss: 3.6861, Val MAE: 3.5479, Val Hausdorff: 5.8661

Epoch 2/100 - Train Loss: 3.1109, Val MAE: 3.3268, Val Hausdorff: 5.3730

Epoch 3/100 - Train Loss: 3.0484, Val MAE: 3.1874, Val Hausdorff: 4.9541

Epoch 4/100 - Train Loss: 2.8484, Val MAE: 1.9274, Val Hausdorff: 2.3965

Epoch 5/100 - Train Loss: 2.7134, Val MAE: 1.7830, Val Hausdorff: 2.1423

Epoch 6/100 - Train Loss: 2.6780, Val MAE: 1.7614, Val Hausdorff: 2.0995

Epoch 7/100 - Train Loss: 2.6211, Val MAE: 1.8326, Val Hausdorff: 2.2109

Epoch 8/100 - Train Loss: 2.5968, Val MAE: 1.7680, Val Hausdorff: 2.0790


KeyboardInterrupt: 

In [ ]:
# Train the model
model = train_vector_conditioned_ddpm(
    T=1000,
    learning_rate=1e-3,
    train_data=train_loader,
    val_data=val_loader,
    test_data=test_loader,
    epochs=100,
    batch_size=64,
    ema=True,
    model_type='pos_frac',
    atom_mapping_path='atom_type_mapping.json'
)

In [6]:
# Train the model
model = train_vector_conditioned_ddpm(
    T=1000,
    learning_rate=1e-3,
    train_data=train_loader,
    val_data=val_loader,
    test_data=test_loader,
    epochs=100,
    batch_size=64,
    ema=True,
    model_type='pos_frac'
)

Using device: cuda:0


Training:   0%|          | 0/3200 [00:00<?, ?it/s]


Epoch 1/100 - Train Loss: 0.9864, Val MAE: 49.0907

Epoch 2/100 - Train Loss: 0.3961, Val MAE: 5.3649

Epoch 3/100 - Train Loss: 0.3090, Val MAE: 2.4365

Epoch 4/100 - Train Loss: 0.2760, Val MAE: 1.8552

Epoch 5/100 - Train Loss: 0.2705, Val MAE: 1.8282

Epoch 6/100 - Train Loss: 0.2668, Val MAE: 1.7875

Epoch 7/100 - Train Loss: 0.2467, Val MAE: 1.7707

Epoch 8/100 - Train Loss: 0.2435, Val MAE: 1.7477

Epoch 9/100 - Train Loss: 0.2518, Val MAE: 1.7486

Epoch 10/100 - Train Loss: 0.2497, Val MAE: 1.7441

Epoch 11/100 - Train Loss: 0.2393, Val MAE: 1.7292

Epoch 12/100 - Train Loss: 0.2395, Val MAE: 1.7212

Epoch 13/100 - Train Loss: 0.2386, Val MAE: 1.7409

Epoch 14/100 - Train Loss: 0.2384, Val MAE: 1.7542

Epoch 15/100 - Train Loss: 0.2344, Val MAE: 1.7247

Epoch 16/100 - Train Loss: 0.2406, Val MAE: 1.7247

Epoch 17/100 - Train Loss: 0.2397, Val MAE: 1.7408

Epoch 18/100 - Train Loss: 0.2448, Val MAE: 1.7254

Epoch 19/100 - Train Loss: 0.2510, Val MAE: 1.7099

Epoch 20/100 - Trai

In [5]:
# Train the model
model = train_vector_conditioned_ddpm(
    T=100,
    learning_rate=1e-3,
    train_data=train_loader,
    val_data=val_loader,
    test_data=test_loader,
    epochs=100,
    batch_size=64,
    ema=True,
    model_type='pos_abs'
)

Using device: cuda:0


Training:   0%|          | 0/3200 [00:00<?, ?it/s]


Epoch 1/100 - Train Loss: 1.2515, Val MAE: 5.9365

Epoch 2/100 - Train Loss: 1.0005, Val MAE: 5.9735

Epoch 3/100 - Train Loss: 0.9965, Val MAE: 5.9417

Epoch 4/100 - Train Loss: 0.9947, Val MAE: 5.8837

Epoch 5/100 - Train Loss: 0.9870, Val MAE: 5.8001

Epoch 6/100 - Train Loss: 0.9775, Val MAE: 5.7917

Epoch 7/100 - Train Loss: 0.9768, Val MAE: 5.7940

Epoch 8/100 - Train Loss: 0.9736, Val MAE: 5.7560

Epoch 9/100 - Train Loss: 0.9749, Val MAE: 5.7253

Epoch 10/100 - Train Loss: 0.9704, Val MAE: 5.7645

Epoch 11/100 - Train Loss: 0.9686, Val MAE: 5.7265

Epoch 12/100 - Train Loss: 0.9646, Val MAE: 5.7297

Epoch 13/100 - Train Loss: 0.9658, Val MAE: 5.7190

Epoch 14/100 - Train Loss: 0.9647, Val MAE: 5.6997

Epoch 15/100 - Train Loss: 0.9572, Val MAE: 5.7059

Epoch 16/100 - Train Loss: 0.9558, Val MAE: 5.7044

Epoch 17/100 - Train Loss: 0.9569, Val MAE: 5.6891

Epoch 18/100 - Train Loss: 0.9509, Val MAE: 5.7109

Epoch 19/100 - Train Loss: 0.9509, Val MAE: 5.6906

Epoch 20/100 - Train

In [5]:
# Train the model
model = train_vector_conditioned_ddpm(
    T=100,
    learning_rate=1e-3,
    train_data=train_loader,
    val_data=val_loader,
    test_data=test_loader,
    epochs=100,
    batch_size=64,
    ema=True,
    model_type='pos_frac'
)

Using device: cuda:0


Training:   0%|          | 0/3200 [00:00<?, ?it/s]


Epoch 1/100 - Train Loss: 1.2335, Val MAE: 3.1951

Epoch 2/100 - Train Loss: 0.8583, Val MAE: 1.9899

Epoch 3/100 - Train Loss: 0.7278, Val MAE: 1.8096

Epoch 4/100 - Train Loss: 0.6921, Val MAE: 1.7876

Epoch 5/100 - Train Loss: 0.6677, Val MAE: 1.7594

Epoch 6/100 - Train Loss: 0.6537, Val MAE: 1.7361

Epoch 7/100 - Train Loss: 0.6561, Val MAE: 1.6905

Epoch 8/100 - Train Loss: 0.6485, Val MAE: 1.7598

Epoch 9/100 - Train Loss: 0.6528, Val MAE: 1.7183

Epoch 10/100 - Train Loss: 0.6440, Val MAE: 1.7244

Epoch 11/100 - Train Loss: 0.6397, Val MAE: 1.7135

Epoch 12/100 - Train Loss: 0.6330, Val MAE: 1.7218

Epoch 13/100 - Train Loss: 0.6396, Val MAE: 1.7370

Epoch 14/100 - Train Loss: 0.6293, Val MAE: 1.6948

Epoch 15/100 - Train Loss: 0.6382, Val MAE: 1.7062

Epoch 16/100 - Train Loss: 0.6320, Val MAE: 1.7182

Epoch 17/100 - Train Loss: 0.6351, Val MAE: 1.7305

Epoch 18/100 - Train Loss: 0.6189, Val MAE: 1.7270

Epoch 19/100 - Train Loss: 0.6245, Val MAE: 1.7321

Epoch 20/100 - Train

In [7]:
# Train the model
model = train_vector_conditioned_ddpm(
    T=1000,
    learning_rate=1e-2,
    train_data=train_loader,
    val_data=val_loader,
    test_data=test_loader,
    epochs=100,
    batch_size=64,
    ema=True,
)

Using device: cuda:0


Training:   0%|          | 0/3200 [00:00<?, ?it/s]

Training complete. Model saved to 'training_samples\T1000_lr0.01_epochs100_batch64_cond64\vector_conditioned_rgb_model.pt'


In [8]:
# Train the model
model = train_vector_conditioned_ddpm(
    T=1000,
    learning_rate=1e-3,
    train_data=train_loader,
    val_data=val_loader,
    test_data=test_loader,
    epochs=300,
    batch_size=64,
    ema=True,
)

Using device: cuda:0


Training:   0%|          | 0/9600 [00:00<?, ?it/s]

Training complete. Model saved to 'training_samples\T1000_lr0.001_epochs300_batch64_cond64\vector_conditioned_rgb_model.pt'


In [9]:
# Train the model
model = train_vector_conditioned_ddpm(
    T=1000,
    learning_rate=1e-3,
    train_data=train_loader,
    val_data=val_loader,
    test_data=test_loader,
    epochs=500,
    batch_size=64,
    ema=True,
)

Using device: cuda:0


Training:   0%|          | 0/16000 [00:00<?, ?it/s]

Training complete. Model saved to 'training_samples\T1000_lr0.001_epochs500_batch64_cond64\vector_conditioned_rgb_model.pt'
